# Async (asynchronous) IO 

It gives a feeling of concurrency despite using a single thread in a single process. It takes long waiting periods in which functions would otherwise be blocking and allows other functions to run during that downtime.

> Normally, a function that blocks effectively forbids others from running from the time that it starts until the time that it returns.

Asyncio, on the other hand, uses **cooperative multitasking**. The tasks must cooperate by announcing when they are ready to be switched out. That means that the code in the task has to change slightly to make this happen.

> There is also **preemptive multitasking** which, unlike **cooperative multitasking**, is the operating system that chooses when to switch threads.

|           Concurrency Type           |                          Switching decision                           | Number of processors |
|:------------------------------------:|:---------------------------------------------------------------------:|:--------------------:|
| Pre-emptive multitasking (threading) | The operating system decides when to switch tasks external to Python. |          1           |
|                                      |                                                                       |                      |
|  Cooperative multitasking (asyncio)  |               The tasks decide when to give up control.               |          1           |
|                                      |                                                                       |                      |
|  Multiprocessing (multiprocessing)   |    The processes all run at the same time on different processors.    |         Many         |


## Example

You neeed to download web pages from a few sites, but it really could be any network traffic.

In [ ]:
!pip install requests asyncio aiohttp

In [ ]:
import aiohttp
import asyncio
import concurrent.futures
import multiprocessing
import requests
import threading
import time

### Synchronous version

In [ ]:
def download_site(url, session):
	with session.get(url) as response:
		print(f"Read {len(response.content)} from {url}")

def download_all_sites(sites):
	with requests.Session() as session:
		for url in sites:
			download_site(url, session)


if __name__ == "__main__":
	sites = [
		"https://www.jython.org",
		"http://olympus.realpython.org/dice",
	] * 80
	start_time = time.time()
	download_all_sites(sites)
	duration = time.time() - start_time
	print(f"Downloaded {len(sites)} in {duration} seconds")

### `threading` version

In [ ]:
thread_local = threading.local()

def get_session():
	if not hasattr(thread_local, "session"):
		thread_local.session = requests.Session()
	return thread_local.session

def download_site(url):
	session = get_session()
	with session.get(url) as response:
		print(f"[{threading.get_ident()}] Read {len(response.content)} from {url}")

def download_all_sites(sites):
	with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
		executor.map(download_site, sites)


if __name__ == "__main__":
	sites = [
		"https://www.jython.org",
		"http://olympus.realpython.org/dice",
	] * 80
	start_time = time.time()
	download_all_sites(sites)
	duration = time.time() - start_time
	print(f"Downloaded {len(sites)} in {duration} seconds")

Threads can interact in ways that are subtle and hard to detect. These interactions can cause race conditions (happen because the programmer has not sufficiently protected data accesses to prevent threads from interfering with each other) that frequently result in random, intermittent bugs that can be quite difficult to find.

## `asyncio` version

In [ ]:
async def download_site(session, url):
	async with session.get(url) as response:
		print("Read {0} from {1}".format(response.content_length, url))

async def download_all_sites(sites):
	async with aiohttp.ClientSession() as session:
		tasks = []
		for url in sites:
			task = asyncio.ensure_future(download_site(session, url))
			tasks.append(task)
		await asyncio.gather(*tasks, return_exceptions=True)


if __name__ == "__main__":
	sites = [
		"https://www.jython.org",
		"http://olympus.realpython.org/dice",
	] * 80
	start_time = time.time()
	asyncio.run(download_all_sites(sites))
	duration = time.time() - start_time
	print(f"Downloaded {len(sites)} sites in {duration} seconds")

A subtle issue is that all of the advantages of cooperative multitasking get thrown away if one of the tasks does not cooperate. A minor mistake in code can cause a task to run off and hold the processor for a long time, starving other tasks that need running.

## `multiprocessing` version

In [ ]:
session = None

def set_global_session():
	global session
	if not session:
		session = requests.Session()

def download_site(url):
	with session.get(url) as response:
		name = multiprocessing.current_process().name
		print(f"{name}:Read {len(response.content)} from {url}")

def download_all_sites(sites):
	with multiprocessing.Pool(initializer=set_global_session) as pool:
		pool.map(download_site, sites)


if __name__ == "__main__":
	sites = [
		"https://www.jython.org",
		"http://olympus.realpython.org/dice",
	] * 80
	start_time = time.time()
	download_all_sites(sites)
	duration = time.time() - start_time
	print(f"Downloaded {len(sites)} in {duration} seconds")